# 30 - Multi-Agent Systems

Multi-agent systems use multiple specialized agents instead of one general-purpose agent.

Common patterns:
- supervisor / worker
- router
- handoff
- parallel specialists
- shared state
- custom graph

The important engineering question is:
> Does separating the roles improve the task enough to justify extra complexity, latency, and cost?

## Setup

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()

CHAT_MODEL = os.getenv("OPENAI_CHAT_MODEL", "")
if CHAT_MODEL:
    print("Configured model:", CHAT_MODEL)
else:
    print("Set OPENAI_CHAT_MODEL for live model examples.")

# %pip install -qU langchain langchain-openai langgraph python-dotenv

## 1. Single agent vs multi-agent

Single agent:
```text
Goal → one agent → answer
```

Multi-agent:
```text
Supervisor
 ├─ Researcher
 ├─ Analyst
 └─ Writer
        ↓
     final answer
```

## 2. Specialist agents

Each specialist should have:
- narrow role
- focused tools/context
- clear input/output contract

In [ ]:
from langchain.agents import create_agent

if CHAT_MODEL:
    research_agent = create_agent(
        model=CHAT_MODEL,
        tools=[],
        system_prompt='You are a research specialist. Return concise evidence.',
    )

    writer_agent = create_agent(
        model=CHAT_MODEL,
        tools=[],
        system_prompt='You are a writing specialist. Produce a clear final answer from evidence.',
    )

    print('Specialist agents ready.')
else:
    print('Configure OPENAI_CHAT_MODEL first.')

## 3. Expose specialists as tools

One simple multi-agent architecture is a supervisor calling subagents as tools.

In [ ]:
from langchain.tools import tool

@tool
def ask_researcher(question: str) -> str:
    'Ask the research specialist for evidence.'
    if not CHAT_MODEL:
        return 'Model not configured.'
    result = research_agent.invoke({
        'messages': [{'role': 'user', 'content': question}]
    })
    return result['messages'][-1].content

@tool
def ask_writer(instructions: str) -> str:
    'Ask the writing specialist to draft a response.'
    if not CHAT_MODEL:
        return 'Model not configured.'
    result = writer_agent.invoke({
        'messages': [{'role': 'user', 'content': instructions}]
    })
    return result['messages'][-1].content

## 4. Supervisor

The supervisor decides which specialist to call and when.

In [ ]:
if CHAT_MODEL:
    supervisor = create_agent(
        model=CHAT_MODEL,
        tools=[ask_researcher, ask_writer],
        system_prompt=(
            'You are a supervisor. Use the researcher for evidence, '
            'then the writer for the final response.'
        ),
    )
    print('Supervisor ready.')

## 5. Router pattern

A router chooses one specialist early.

```text
input
 ↓
classifier/router
 ├─ coding
 ├─ research
 └─ support
```

## 6. Parallel specialists

Independent subtasks can sometimes run in parallel.

Example:
```text
question
 ├─ research pricing
 ├─ research specs
 └─ research reviews
       ↓
     combine
```

## 7. Shared state

Multi-agent systems need clear boundaries around shared state.

Do not let every agent freely overwrite everything.

In [ ]:
shared_state = {
    'goal': 'compare two AI systems',
    'research': [],
    'analysis': [],
    'draft': '',
}

print(shared_state)

## 8. Multi-agent evaluation

Measure:
- routing accuracy
- subtask success
- final task success
- redundant calls
- latency
- cost
- disagreement/reconciliation quality

Always compare against a strong single-agent baseline.

## 9. When NOT to use multi-agent

Do not create five agents for a problem one tool call can solve.

Multi-agent systems make sense when:
- roles are genuinely distinct
- tasks can be parallelized
- different tools/context are needed
- specialization materially improves quality

## 10. Practice

Build:
1. researcher
2. analyst
3. writer
4. supervisor

Then compare:
- single agent
- three specialists
on the same benchmark.

## Evaluation

Evaluation should include both quality and systems cost.

Improve by:
- narrow agent roles
- explicit contracts
- controlled delegation
- parallelism where safe
- state boundaries
- regression tests.

## References / current documentation


- LangChain reference: https://reference.langchain.com/python/langchain/overview
- LangGraph reference: https://reference.langchain.com/python/langgraph/overview
- LangGraph persistence: https://github.com/langchain-ai/docs/blob/main/src/oss/langgraph/persistence.mdx
- LangGraph interrupts: https://github.com/langchain-ai/docs/blob/main/src/oss/langgraph/interrupts.mdx
- LangChain human-in-the-loop: https://github.com/langchain-ai/docs/blob/main/src/oss/langchain/human-in-the-loop.mdx
- Ragas metrics: https://docs.ragas.io/en/latest/concepts/metrics/available_metrics/
- MCP Python SDK v2: https://py.sdk.modelcontextprotocol.io/
- MCP Python SDK getting started: https://py.sdk.modelcontextprotocol.io/get-started/